In [1]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_available())

NVIDIA GeForce RTX 5080
True


In [2]:
import os
import re
import pandas as pd
import numpy as np
import torch
import torchaudio
import miniaudio
import evaluate
from datasets import Dataset, Features, Value
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
    )

DATA_DIR = "/home/toru2/Amara/Deep_learning/lab3/common_voice_mn"
CLIPS_DIR = os.path.join(DATA_DIR, "clips")

SEED = 42
set_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("seed:", SEED)

def load_audio_mono(path: str, target_sr: int = 16000) -> tuple[np.ndarray, int]:
    """Return (mono_float32_waveform, sampling_rate)."""
    try:
        waveform, sr = torchaudio.load(path)
        if waveform.ndim == 2 and waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        waveform = waveform.squeeze(0).to(torch.float32)
        if sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, target_sr)
            sr = target_sr
        return waveform.cpu().numpy(), sr
    except Exception:
        pass

    decoded = miniaudio.decode_file(path)
    sr = int(decoded.sample_rate)
    samples = np.asarray(decoded.samples)
    if decoded.nchannels and decoded.nchannels > 1:
        samples = samples.reshape(-1, int(decoded.nchannels)).mean(axis=1)
    samples = samples.astype(np.float32)
    if sr != target_sr:
        # lightweight linear resample (keeps deps minimal)
        x_old = np.linspace(0.0, 1.0, num=len(samples), endpoint=False)
        n_new = int(round(len(samples) * (target_sr / sr)))
        x_new = np.linspace(0.0, 1.0, num=max(n_new, 1), endpoint=False)
        samples = np.interp(x_new, x_old, samples).astype(np.float32)
        sr = target_sr
    return samples, sr

# ---------- Load local TSVs ----------
def load_tsv(split_name: str) -> Dataset:
    tsv_path = os.path.join(DATA_DIR, f"{split_name}.tsv")
    df = pd.read_csv(tsv_path, sep="\t", usecols=["path", "sentence"])
    df = df.dropna(subset=["path", "sentence"]).copy()
    df["path"] = df["path"].astype(str).str.strip().map(lambda p: os.path.join(CLIPS_DIR, p))
    df["sentence"] = df["sentence"].astype(str)
    
    # Force Arrow schema to plain strings (avoids large_string/dictionary issues)
    features = Features({"path": Value("string"), "sentence": Value("string")})
    return Dataset.from_pandas(df, features=features, preserve_index=False)

ds_train = load_tsv("train")
ds_dev = load_tsv("dev")
ds_test = load_tsv("test")

print("train:", ds_train)
print("dev:", ds_dev)
print("test:", ds_test)

device: cuda
seed: 42
train: Dataset({
    features: ['path', 'sentence'],
    num_rows: 2193
})
dev: Dataset({
    features: ['path', 'sentence'],
    num_rows: 1897
})
test: Dataset({
    features: ['path', 'sentence'],
    num_rows: 1936
})


In [3]:
from pathlib import Path

def _load_env_file(env_path: str) -> dict:
    env = {}
    p = Path(env_path)
    if not p.is_file():
        return env
    for raw_line in p.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        if line.startswith("export "):
            line = line[len("export "):].lstrip()
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip()
        v = v.strip('"').strip("'")
        if " #" in v:
            v = v.split(" #", 1)[0].rstrip()
        if k:
            env[k] = v
    return env

ENV_PATH = "/home/toru2/Amara/Deep_learning/lab3/.env"
env_vars = _load_env_file(ENV_PATH)

for k, v in env_vars.items():
    if k in {"HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"}:
        os.environ[k] = v
    else:
        os.environ.setdefault(k, v)

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if HF_TOKEN:
    HF_TOKEN = HF_TOKEN.strip()
    if HF_TOKEN.lower().startswith("bearer "):
        HF_TOKEN = HF_TOKEN.split(None, 1)[1].strip()
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

print(".env loaded:", bool(env_vars))
print("HF_TOKEN present:", bool(HF_TOKEN))
if HF_TOKEN:
    print("HF_TOKEN length:", len(HF_TOKEN))
    print("HF_TOKEN looks like hf_...:", HF_TOKEN.startswith("hf_"))

.env loaded: True
HF_TOKEN present: True
HF_TOKEN length: 37
HF_TOKEN looks like hf_...: True


In [4]:
# ---------- Sanity check 1: files exist ----------
def path_exists(p: str) -> bool:
    try:
        return os.path.isfile(p)
    except Exception:
        return False

missing_train = [p for p in ds_train["path"][:200] if not path_exists(p)]
missing_dev = [p for p in ds_dev["path"][:200] if not path_exists(p)]
missing_test = [p for p in ds_test["path"][:200] if not path_exists(p)]
print("missing train (first 200):", len(missing_train))
print("missing dev   (first 200):", len(missing_dev))
print("missing test  (first 200):", len(missing_test))
if missing_train[:3]:
    print("example missing train paths:", missing_train[:3])
if missing_dev[:3]:
    print("example missing dev paths:", missing_dev[:3])
if missing_test[:3]:
    print("example missing test paths:", missing_test[:3])

# Optional: filter out missing files globally
ds_train = ds_train.filter(lambda x: path_exists(x["path"]))
ds_dev = ds_dev.filter(lambda x: path_exists(x["path"]))
ds_test = ds_test.filter(lambda x: path_exists(x["path"]))
print("after filtering:", ds_train, ds_dev, ds_test)

missing train (first 200): 0
missing dev   (first 200): 0
missing test  (first 200): 0


Filter:   0%|          | 0/2193 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1936 [00:00<?, ? examples/s]

after filtering: Dataset({
    features: ['path', 'sentence'],
    num_rows: 2193
}) Dataset({
    features: ['path', 'sentence'],
    num_rows: 1897
}) Dataset({
    features: ['path', 'sentence'],
    num_rows: 1936
})


In [5]:
# ---------- Sanity check 2: load one audio file ----------
p0 = ds_train[0]["path"]
audio, sr = load_audio_mono(p0, target_sr=16000)
print("path:", p0)
print("sampling rate:", sr)
print("mono shape:", audio.shape)
print("seconds:", float(len(audio) / sr))

path: /home/toru2/Amara/Deep_learning/lab3/common_voice_mn/clips/common_voice_mn_21799896.mp3
sampling rate: 16000
mono shape: (103296,)
seconds: 6.456


In [6]:
# ---------- Text normalization policy (enforced for train/dev/test) ----------
import unicodedata

TRANSCRIPT_CHAR_MAP = str.maketrans({
    "\u2018": "'",
    "\u2019": "'",
    "\u201c": '"',
    "\u201d": '"',
    "\u2013": "-",
    "\u2014": "-",
    "\u2212": "-",
    "\u00a0": " ",
    "\u2009": " ",
    "\u200a": " ",
    "\u200b": "",
    "\ufeff": "",
})

# Keep Mongolian letters, remove punctuation, and normalize spaces.
chars_to_ignore_regex = r"[,\?\.\!\-\;\:\"\“\”\’\‘\'\(\)\[\]\{\}…]"

def normalize_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(TRANSCRIPT_CHAR_MAP)
    s = s.lower()
    s = re.sub(chars_to_ignore_regex, " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

print("before:", ds_train[0]["sentence"])
print("after :", normalize_text(ds_train[0]["sentence"]))

# Filter out empty normalized transcripts
ds_train = ds_train.filter(lambda x: len(normalize_text(x["sentence"])) > 0)
ds_dev = ds_dev.filter(lambda x: len(normalize_text(x["sentence"])) > 0)
ds_test = ds_test.filter(lambda x: len(normalize_text(x["sentence"])) > 0)
print("after text filtering:", ds_train, ds_dev, ds_test)

before: хэмээн өөрийгөө тойрч зогссон шавь нартаа туньсан янзтай уйлж байлаа.
after : хэмээн өөрийгөө тойрч зогссон шавь нартаа туньсан янзтай уйлж байлаа


Filter:   0%|          | 0/2193 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1936 [00:00<?, ? examples/s]

after text filtering: Dataset({
    features: ['path', 'sentence'],
    num_rows: 2192
}) Dataset({
    features: ['path', 'sentence'],
    num_rows: 1897
}) Dataset({
    features: ['path', 'sentence'],
    num_rows: 1936
})


In [7]:
# ---------- Load processor/model + resampler ----------
# This notebook runs in an environment where anonymous HF downloads may fail, so we use HF_TOKEN.
# Also: some community Wav2Vec2 repos do NOT include `processor_config.json` (Transformers 5.x will 404),
# so we build the processor from tokenizer + feature extractor in that case.

from huggingface_hub import HfApi
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

# Pick the correct repo id. The short id in some model cards is ambiguous; these are known working repos.
MODEL_ID = "wav2vec2-large-xlsr-53-mongolian"  # you can also set this to a full repo id like "tugstugi/wav2vec2-large-xlsr-53-mongolian"
KNOWN_MODEL_ALIASES = {
    "wav2vec2-large-xlsr-53-mongolian": "tugstugi/wav2vec2-large-xlsr-53-mongolian",
}

def resolve_model_id(model_id: str) -> str:
    model_id = str(model_id).strip()
    if os.path.isdir(model_id):
        return model_id
    if model_id in KNOWN_MODEL_ALIASES:
        return KNOWN_MODEL_ALIASES[model_id]
    return model_id

resolved_id = resolve_model_id(MODEL_ID)
print("Model id:", resolved_id)

hub_token = None
if not os.path.isdir(resolved_id):
    if not HF_TOKEN:
        raise RuntimeError(
            "HF_TOKEN is missing. Put it in /home/toru2/Amara/Deep_learning/lab3/.env as HF_TOKEN=... and re-run Cell 3."
        )
    hub_token = HF_TOKEN.strip()
    if hub_token.lower().startswith("bearer "):
        hub_token = hub_token.split(None, 1)[1].strip()

    # Verify token early
    who = HfApi().whoami(token=hub_token)
    print("HF whoami:", who.get("name") or who.get("fullname") or "ok")

# Load processor/model
if os.path.isdir(resolved_id):
    # Local folder case
    processor = Wav2Vec2Processor.from_pretrained(resolved_id)
    model = Wav2Vec2ForCTC.from_pretrained(resolved_id).to(device)
else:
    # HF Hub case
    model = Wav2Vec2ForCTC.from_pretrained(resolved_id, token=hub_token).to(device)
    try:
        # Some repos include processor_config.json; if so, this works.
        processor = Wav2Vec2Processor.from_pretrained(resolved_id, token=hub_token)
    except Exception as e:
        # Fallback for repos that only ship preprocessor_config.json + vocab.json
        feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(resolved_id, token=hub_token)
        tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(resolved_id, token=hub_token)
        processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
        print("Processor fallback: built from feature_extractor + tokenizer (no processor_config.json)")

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.ctc_zero_infinity = True
print("pad_token_id:", processor.tokenizer.pad_token_id)

# Your requested resampler (useful only if you *know* your audio is 48kHz)
resampler = torchaudio.transforms.Resample(48_000, 16_000)
print("resampler:", resampler)

Model id: tugstugi/wav2vec2-large-xlsr-53-mongolian
HF whoami: Bokhbat


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

pad_token_id: 37
resampler: Resample()


In [8]:
# --- Verify model + processor are really loaded ---
print('processor type:', type(processor))
print('model type:', type(model))
print('vocab size:', len(processor.tokenizer))
print('model vocab size:', int(model.config.vocab_size))
print('device:', next(model.parameters()).device)

# tiny forward pass sanity check
import numpy as np
x = np.zeros(16000, dtype=np.float32)  # 1 second of silence @16k
inputs = processor(x, sampling_rate=16000, return_tensors='pt')
with torch.no_grad():
    logits = model(inputs.input_values.to(device)).logits
print('logits shape:', tuple(logits.shape))

processor type: <class 'transformers.models.wav2vec2.processing_wav2vec2.Wav2Vec2Processor'>
model type: <class 'transformers.models.wav2vec2.modeling_wav2vec2.Wav2Vec2ForCTC'>
vocab size: 40
model vocab size: 38
device: cuda:0
logits shape: (1, 49, 38)


In [9]:
# ---------- Transcript/tokenizer coverage audit ----------
def collect_chars(ds: Dataset) -> set[str]:
    all_chars = set()
    for text in ds["sentence"]:
        all_chars.update(set(normalize_text(text)))
    return all_chars

tokenizer_vocab = processor.tokenizer.get_vocab()
tokenizer_char_tokens = {tok for tok in tokenizer_vocab if len(tok) == 1}
tokenizer_char_tokens.add(" ")  # tokenizer handles spaces via delimiter token internally

def split_unknown_chars(ds: Dataset, split_name: str) -> list[str]:
    chars = collect_chars(ds)
    unknown = sorted(ch for ch in chars if ch not in tokenizer_char_tokens)
    print(f"{split_name}: {len(chars)} unique normalized chars, unknown={len(unknown)}")
    if unknown:
        print(f"{split_name} unknown chars:", unknown)
    return unknown

unknown_train = split_unknown_chars(ds_train, "train")
unknown_dev = split_unknown_chars(ds_dev, "dev")
unknown_test = split_unknown_chars(ds_test, "test")

all_unknown = sorted(set(unknown_train) | set(unknown_dev) | set(unknown_test))
print("all unknown chars:", all_unknown)

def preview_unknown_examples(ds: Dataset, unknown_chars: list[str], limit: int = 5):
    if not unknown_chars:
        print("No unknown-char examples found.")
        return
    u = set(unknown_chars)
    shown = 0
    for text in ds["sentence"]:
        n = normalize_text(text)
        bad = sorted(set(n) & u)
        if bad:
            print("unknown chars:", bad)
            print("normalized:", n)
            print("-" * 80)
            shown += 1
            if shown >= limit:
                break

preview_unknown_examples(ds_train, all_unknown, limit=5)

if processor.tokenizer.unk_token_id is not None:
    def estimate_unk_rate(ds: Dataset, n: int = 1000) -> float:
        n = min(n, len(ds))
        total_ids = 0
        unk_ids = 0
        for text in ds["sentence"][:n]:
            ids = processor.tokenizer(normalize_text(text)).input_ids
            total_ids += len(ids)
            unk_ids += sum(i == processor.tokenizer.unk_token_id for i in ids)
        return 0.0 if total_ids == 0 else unk_ids / total_ids

    print("UNK token rate (train sample):", round(100 * estimate_unk_rate(ds_train, n=1000), 4), "%")
else:
    print("Tokenizer has no unk_token_id; skipping UNK-rate estimate.")

train: 66 unique normalized chars, unknown=30
train unknown chars: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'i', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', '«', '»']
dev: 65 unique normalized chars, unknown=30
dev unknown chars: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'i', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', '«', '»']
test: 50 unique normalized chars, unknown=14
test unknown chars: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'i', 'l', 'n', 'o', 'r', 't', 'x']
all unknown chars: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', '«', '»']
unknown chars: ['«', '»']
normalized: бид рашааны овоон дээр очиж томчуудад аашлуулан өргөсөн мөнгө түүж хэрдээ л «баяждаг» байж
--------------------------------------------------------------------------------
unknown chars: ['«', '»']
normalized:

In [10]:
# ---------- Step 3: filter bad/unstable samples ----------
# Decision rule: if loss goes down but WER stalls, tighten these filters first.
MIN_AUDIO_SEC = 0.5
MAX_AUDIO_SEC = 15.0
MAX_TRANSCRIPT_CHARS = 180
MIN_CHARS_PER_SEC = 2.0
MAX_CHARS_PER_SEC = 35.0

def build_quality_metadata(batch):
    text = normalize_text(batch["sentence"])

    try:
        audio, sr = load_audio_mono(batch["path"], target_sr=16000)
        duration_sec = float(len(audio) / sr) if sr > 0 else 0.0
        decode_ok = True
    except Exception:
        duration_sec = -1.0
        decode_ok = False

    transcript_chars = len(text)
    chars_per_sec = float(transcript_chars / duration_sec) if duration_sec > 0 else 0.0

    batch["norm_sentence"] = text
    batch["duration_sec"] = duration_sec
    batch["transcript_chars"] = transcript_chars
    batch["chars_per_sec"] = chars_per_sec
    batch["decode_ok"] = decode_ok
    return batch

def apply_quality_filters(ds: Dataset, split_name: str) -> Dataset:
    before_n = len(ds)
    ds_q = ds.map(build_quality_metadata)

    decode_ok_col = ds_q["decode_ok"]
    duration_col = ds_q["duration_sec"]
    tchars_col = ds_q["transcript_chars"]
    cps_col = ds_q["chars_per_sec"]

    decode_fail = 0
    too_short = 0
    too_long = 0
    empty_text = 0
    too_long_text = 0
    too_slow = 0
    too_fast = 0
    keep_indices = []

    for i in range(len(ds_q)):
        ok = bool(decode_ok_col[i])
        dur = float(duration_col[i])
        tlen = int(tchars_col[i])
        cps = float(cps_col[i])

        if not ok:
            decode_fail += 1
        if ok and dur < MIN_AUDIO_SEC:
            too_short += 1
        if ok and dur > MAX_AUDIO_SEC:
            too_long += 1
        if tlen == 0:
            empty_text += 1
        if tlen > MAX_TRANSCRIPT_CHARS:
            too_long_text += 1
        if ok and cps < MIN_CHARS_PER_SEC:
            too_slow += 1
        if ok and cps > MAX_CHARS_PER_SEC:
            too_fast += 1

        if (
            ok
            and (MIN_AUDIO_SEC <= dur <= MAX_AUDIO_SEC)
            and (0 < tlen <= MAX_TRANSCRIPT_CHARS)
            and (MIN_CHARS_PER_SEC <= cps <= MAX_CHARS_PER_SEC)
        ):
            keep_indices.append(i)

    ds_q = ds_q.select(keep_indices)
    after_n = len(ds_q)

    print(f"[{split_name}] before={before_n}, after={after_n}, kept={after_n / max(before_n, 1):.2%}")
    print(
        f"[{split_name}] decode_fail={decode_fail}, too_short={too_short}, too_long={too_long}, "
        f"empty_text={empty_text}, too_long_text={too_long_text}, too_slow={too_slow}, too_fast={too_fast}"
    )

    # Keep canonical fields only for downstream pipeline
    return ds_q.remove_columns([
        "norm_sentence",
        "duration_sec",
        "transcript_chars",
        "chars_per_sec",
        "decode_ok",
    ])

ds_train = apply_quality_filters(ds_train, "train")
ds_dev = apply_quality_filters(ds_dev, "dev")
ds_test = apply_quality_filters(ds_test, "test")

print("after quality filtering:", ds_train, ds_dev, ds_test)

Map:   0%|          | 0/2192 [00:00<?, ? examples/s]

[train] before=2192, after=2190, kept=99.91%
[train] decode_fail=0, too_short=1, too_long=0, empty_text=0, too_long_text=1, too_slow=0, too_fast=2


Map:   0%|          | 0/1897 [00:00<?, ? examples/s]

[dev] before=1897, after=1895, kept=99.89%
[dev] decode_fail=0, too_short=0, too_long=0, empty_text=0, too_long_text=2, too_slow=0, too_fast=2


Map:   0%|          | 0/1936 [00:00<?, ? examples/s]

[test] before=1936, after=1936, kept=100.00%
[test] decode_fail=0, too_short=0, too_long=0, empty_text=0, too_long_text=0, too_slow=0, too_fast=0
after quality filtering: Dataset({
    features: ['path', 'sentence'],
    num_rows: 2190
}) Dataset({
    features: ['path', 'sentence'],
    num_rows: 1895
}) Dataset({
    features: ['path', 'sentence'],
    num_rows: 1936
})


In [11]:
# ---------- Audio -> input_values + labels (start with a small subset) ----------
def speech_file_to_array(path: str, target_sr: int = 16000) -> np.ndarray:
    audio, sr = load_audio_mono(path, target_sr=target_sr)
    # already mono float32
    return audio.astype(np.float32)

def prepare_batch(batch):
    text = normalize_text(batch["sentence"])
    speech = speech_file_to_array(batch["path"], target_sr=16000)
    inputs = processor(speech, sampling_rate=16000)
    batch["input_values"] = inputs.input_values[0]
    # Transformers 5.x: Wav2Vec2Processor may not provide as_target_processor()
    batch["labels"] = processor.tokenizer(text).input_ids
    return batch

mini_train = ds_train.select(range(min(50, len(ds_train)))).map(prepare_batch, remove_columns=ds_train.column_names)
print(mini_train)
print("keys:", mini_train[0].keys())
print("len(input_values):", len(mini_train[0]["input_values"]))
print("len(labels):", len(mini_train[0]["labels"]))

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Dataset({
    features: ['input_values', 'labels'],
    num_rows: 50
})
keys: dict_keys(['input_values', 'labels'])
len(input_values): 103296
len(labels): 68


In [12]:
# ---------- Data collator sanity check ----------
from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        # Transformers 5.x: avoid processor.as_target_processor(); pad labels with tokenizer directly
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

collator = DataCollatorCTCWithPadding(processor)
b = collator([mini_train[i] for i in range(min(4, len(mini_train)))])
print({k: v.shape for k, v in b.items()})
print("labels contain -100?", bool((b["labels"] == -100).any().item()))

{'input_values': torch.Size([4, 107904]), 'attention_mask': torch.Size([4, 107904]), 'labels': torch.Size([4, 93])}
labels contain -100? True


In [13]:
# ---------- One forward pass sanity check (loss finite) ----------
model.train()
out = model(
    input_values=b["input_values"].to(device),
    attention_mask=b["attention_mask"].to(device),
    labels=b["labels"].to(device),
 )
print("loss:", float(out.loss))
assert torch.isfinite(out.loss).item()

loss: 1.1932169198989868


/tmp/ipykernel_221151/807372650.py:8: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("loss:", float(out.loss))


In [14]:
# ---------- Step 4: strong baseline fast loop (fixed small-dev + optional full-dev) ----------
wer_metric = evaluate.load("wer")
model.eval()

# Fixed subset for quick iteration (same samples every run for fair comparison).
FAST_DEV_SIZE = min(300, len(ds_dev))
FAST_DEV_SEED = 42
rng = np.random.default_rng(FAST_DEV_SEED)
fast_dev_indices = np.sort(rng.choice(len(ds_dev), size=FAST_DEV_SIZE, replace=False))
fast_dev_set = ds_dev.select(fast_dev_indices.tolist())

# Only compute full-dev when fast-dev improves enough.
FAST_DEV_IMPROVEMENT_THRESHOLD = 0.005  # 0.5 absolute WER points (fraction scale)
if "_baseline_state" not in globals():
    _baseline_state = {
        "best_fast_wer": None,
        "last_full_wer": None,
    }

# Cache expensive eval preprocessing by dataset fingerprint.
if "_eval_prepared_cache" not in globals():
    _eval_prepared_cache = {}

def get_prepared_for_eval(ds: Dataset) -> Dataset:
    key = ds._fingerprint
    if key not in _eval_prepared_cache:
        _eval_prepared_cache[key] = ds.map(prepare_batch, remove_columns=ds.column_names)
    return _eval_prepared_cache[key]

def decode_labels(label_ids: np.ndarray) -> list[str]:
    label_ids = label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    return processor.batch_decode(label_ids, group_tokens=False)

def predict_wer(ds: Dataset, batch_size: int = 8) -> float:
    ds_p = get_prepared_for_eval(ds)
    n = len(ds_p)
    preds, refs = [], []
    for start in range(0, n, batch_size):
        feats = [ds_p[i] for i in range(start, min(start + batch_size, n))]
        bb = collator(feats)
        with torch.no_grad():
            logits = model(
                input_values=bb["input_values"].to(device),
                attention_mask=bb["attention_mask"].to(device),
            ).logits
        pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
        pred_str = processor.batch_decode(pred_ids)
        ref_str = decode_labels(bb["labels"].cpu().numpy())
        preds.extend([normalize_text(s) for s in pred_str])
        refs.extend([normalize_text(s) for s in ref_str])
    return wer_metric.compute(predictions=preds, references=refs)

# Quick baseline on fixed small-dev subset
fast_dev_wer = predict_wer(fast_dev_set, batch_size=8)
print(f"FAST dev WER ({FAST_DEV_SIZE} clips, fixed subset): {100 * fast_dev_wer:.2f}%")

run_full_dev = False
if _baseline_state["best_fast_wer"] is None:
    run_full_dev = True
elif (_baseline_state["best_fast_wer"] - fast_dev_wer) >= FAST_DEV_IMPROVEMENT_THRESHOLD:
    run_full_dev = True

# Update best fast-dev tracker
if _baseline_state["best_fast_wer"] is None or fast_dev_wer < _baseline_state["best_fast_wer"]:
    _baseline_state["best_fast_wer"] = fast_dev_wer

print(f"Best FAST dev WER so far: {100 * _baseline_state['best_fast_wer']:.2f}%")
print("Run full dev now:", run_full_dev)

if run_full_dev:
    full_dev_wer = predict_wer(ds_dev, batch_size=8)
    _baseline_state["last_full_wer"] = full_dev_wer
    print(f"FULL dev WER ({len(ds_dev)} clips): {100 * full_dev_wer:.2f}%")
else:
    print("Skipping full-dev WER this round (fast-dev improvement not large enough).")
    if _baseline_state["last_full_wer"] is not None:
        print(f"Last FULL dev WER: {100 * _baseline_state['last_full_wer']:.2f}%")

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

FAST dev WER (300 clips, fixed subset): 27.33%
Best FAST dev WER so far: 27.33%
Run full dev now: True


Map:   0%|          | 0/1895 [00:00<?, ? examples/s]

FULL dev WER (1895 clips): 29.00%


In [ ]:
# ---------- Training cell (train on train, validate on dev, report test) ----------
ds_train_p = ds_train.map(prepare_batch, remove_columns=ds_train.column_names)
ds_dev_p = ds_dev.map(prepare_batch, remove_columns=ds_dev.column_names)
ds_test_p = ds_test.map(prepare_batch, remove_columns=ds_test.column_names)

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

training_args = TrainingArguments(
    output_dir="./wav2vec2-mn-ft",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    do_train=True,
    do_eval=True,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=500,
    eval_steps=500,
    logging_steps=50,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    # group_by_length=True,
    num_train_epochs=10,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    data_seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
 )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train_p,
    eval_dataset=ds_dev_p,
    data_collator=collator,
    processing_class=processor,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
 )

train_output = trainer.train()
print(train_output)

# Callback-free WER checks for notebook compatibility
dev_wer = predict_wer(ds_dev, batch_size=8)
test_wer = predict_wer(ds_test, batch_size=8)
print({"dev_wer": dev_wer, "test_wer": test_wer})

# Save final teacher-ready artifact
FINAL_DIR = "./wav2vec2-mn-ft/final-best"
os.makedirs(FINAL_DIR, exist_ok=True)
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print("saved:", FINAL_DIR)

Map:   0%|          | 0/2190 [00:00<?, ? examples/s]

Map:   0%|          | 0/1895 [00:00<?, ? examples/s]

Map:   0%|          | 0/1936 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 39, 'bos_token_id': 38}.


{'loss': '1.285', 'grad_norm': '3.585', 'learning_rate': '7.153e-06', 'epoch': '0.365'}
{'loss': '0.8896', 'grad_norm': '2.29', 'learning_rate': '1.445e-05', 'epoch': '0.7299'}
{'loss': '0.7881', 'grad_norm': '1.526', 'learning_rate': '2e-05', 'epoch': '1.095'}
{'loss': '0.816', 'grad_norm': '2.253', 'learning_rate': '1.988e-05', 'epoch': '1.46'}
{'loss': '0.7879', 'grad_norm': '3.046', 'learning_rate': '1.96e-05', 'epoch': '1.825'}
{'loss': '0.7572', 'grad_norm': '1.629', 'learning_rate': '1.916e-05', 'epoch': '2.19'}
{'loss': '0.7236', 'grad_norm': '1.657', 'learning_rate': '1.858e-05', 'epoch': '2.555'}
{'loss': '0.7263', 'grad_norm': '2.62', 'learning_rate': '1.785e-05', 'epoch': '2.92'}
{'loss': '0.6828', 'grad_norm': '2.214', 'learning_rate': '1.7e-05', 'epoch': '3.285'}
{'loss': '0.7106', 'grad_norm': '1.54', 'learning_rate': '1.604e-05', 'epoch': '3.65'}
{'eval_loss': '0.3236', 'eval_wer': '0.3515', 'eval_runtime': '17.64', 'eval_samples_per_second': '107.4', 'eval_steps_per_se

...

Map:   0%|          | 0/1936 [00:00<?, ? examples/s]

{'dev_wer': 0.27887717788771776, 'test_wer': 0.40971551261406336}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved: ./wav2vec2-mn-ft/final-best


: 

In [ ]:
# ---------- Model check cell (quick qualitative + quantitative checks) ----------
model.eval()

# 1) Callback-free metrics check
if "trainer" in globals():
    dev_wer = predict_wer(ds_dev, batch_size=8)
    test_wer = predict_wer(ds_test, batch_size=8)
    print({"dev_wer": dev_wer, "test_wer": test_wer})
else:
    print("Run the training cell first to create trainer.")

# 2) Show sample predictions on fixed test samples
CHECK_N = min(5, len(ds_test))
for i in range(CHECK_N):
    item = ds_test[i]
    audio_arr = speech_file_to_array(item["path"], target_sr=16000)
    inputs = processor(audio_arr, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(
            input_values=inputs.input_values.to(device),
            attention_mask=inputs.attention_mask.to(device) if "attention_mask" in inputs else None,
        ).logits
    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = normalize_text(processor.batch_decode(pred_ids)[0])
    ref_text = normalize_text(item["sentence"])

    print(f"Sample {i}")
    print("REF :", ref_text)
    print("PRED:", pred_text)
    print("-" * 80)